In [86]:
import geopandas as gpd
from rasterstats import zonal_stats
import fiona
from pathlib import Path
import sys
from utils.gbif_process import (
    gbif_occurrence_search,
    gbif_fetch_all,
    gbif_to_dataframe,
    gbif_to_csv
)

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

In [87]:
# Cargar Capas de la base de datso Geoespacial
gdb_path = "data/boundaries/hnd_admin_boundaries.gdb"
capas = fiona.listlayers(gdb_path)

#Mostrar capas
#Acorde a la metadata la capa admin1 tiene departemtnos. y admin2 muncipios
#
for i, capa in enumerate(capas):
    print(f"[{i}] {capa}")
    
deptos = gpd.read_file(gdb_path, layer=capas[1])

tif_path = "data/Honduras_GISdata_LTAy_YearlyMonthlyTotals_GlobalSolarAtlas-v2_GEOTIFF/GHI.tif"

[0] hnd_admin0
[1] hnd_admin1
[2] hnd_admin2
[3] hnd_adminlines
[4] hnd_adminpoints
[5] hnd_populatedplaces


In [88]:
stats = zonal_stats(
    vectors=deptos, 
    raster=tif_path, 
    stats=["mean", "min", "max", "count"], 
    geojson_out=True,
    all_touched=False # False asegura precisión estricta: el centro del píxel debe estar dentro de la frontera
)

In [89]:
# 4. Transformar los resultados en un GeoDataFrame estructurado
resultados_ghi = gpd.GeoDataFrame.from_features(stats)

#Revisar las columnas de la layer
print(resultados_ghi.columns)

# Mostrar el departamento y su promedio exacto de irradiación
print(resultados_ghi[['adm1_name','mean', 'max']])

Index(['geometry', 'adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3',
       'adm1_pcode', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3',
       'adm0_pcode', 'valid_on', 'valid_to', 'area_sqkm', 'version', 'lang',
       'lang1', 'lang2', 'lang3', 'adm1_ref_name', 'center_lat', 'center_lon',
       'min', 'max', 'mean', 'count'],
      dtype='str')
            adm1_name         mean          max
0   Islas de La Bahia  2006.418033  2054.895996
1               Colon  1848.761151  2042.113037
2           Atlantida  1822.960095  2034.077026
3              Cortes  1885.121888  2032.250977
4                Yoro  1880.036303  2036.269043
5             Olancho  1825.021989  1976.001953
6       Santa Barbara  1864.470463  2005.222046
7               Copan  1836.844392  2053.070068
8             Lempira  2004.704613  2219.259033
9           Comayagua  1932.518827  2166.663086
10           Intibuca  1975.523564  2221.449951
11  Francisco Morazan  1928.571081  2191.864990
12         O

In [ ]:
import geopandas as gpd
from shapely.geometry.polygon import orient

gdb_path = "data/boundaries/hnd_admin_boundaries.gdb"

muni = gpd.read_file(
    gdb_path,
    layer=capas[2]
)

muni = muni.to_crs(4326)

puerto_cortes = muni[
    muni["adm2_name"] == "Choloma"
].copy()

# Simplificar
puerto_cortes["geometry"] = puerto_cortes.geometry.simplify(
    tolerance=0.001,
    preserve_topology=True
)

# Obtener WKT

geom = puerto_cortes.geometry.iloc[0]
geom = orient(geom, sign=1.0)

print(geom.geom_type)
print(geom.is_valid)

wkt = geom.wkt

records = gbif_fetch_all(
    wkt=wkt,
    page_size=300
)




Polygon
True
Downloaded 300 / 6,727 records
Downloaded 600 / 6,727 records
Downloaded 900 / 6,727 records
Downloaded 1,200 / 6,727 records
Downloaded 1,500 / 6,727 records
Downloaded 1,800 / 6,727 records
Downloaded 2,100 / 6,727 records
Downloaded 2,400 / 6,727 records
Downloaded 2,700 / 6,727 records
Downloaded 3,000 / 6,727 records
Downloaded 3,300 / 6,727 records
Downloaded 3,600 / 6,727 records
Downloaded 3,900 / 6,727 records
Downloaded 4,200 / 6,727 records
Downloaded 4,500 / 6,727 records
Downloaded 4,800 / 6,727 records
Downloaded 5,100 / 6,727 records
Downloaded 5,400 / 6,727 records
Downloaded 5,700 / 6,727 records
Downloaded 6,000 / 6,727 records
Downloaded 6,300 / 6,727 records
Downloaded 6,600 / 6,727 records
Downloaded 6,727 / 6,727 records


OSError: Cannot save file into a non-existent directory: 'data/gbif'

In [93]:
df = gbif_to_csv(
    records,
    "data/gbif/puerto_cortes.csv"
)

print(df.head())

Saved 6,727 records to data/gbif/puerto_cortes.csv
          key                            datasetKey  \
0  6163058661  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
1  6178462548  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
2  6171239363  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
3  6179469443  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
4  6251540211  50c9509d-22c7-4a22-a47d-8c48425ef4a7   

                       publishingOrgKey   datasetCategory  \
0  28eb1a3f-1c15-4a95-931a-4af90ecb574d  [CitizenScience]   
1  28eb1a3f-1c15-4a95-931a-4af90ecb574d  [CitizenScience]   
2  28eb1a3f-1c15-4a95-931a-4af90ecb574d  [CitizenScience]   
3  28eb1a3f-1c15-4a95-931a-4af90ecb574d  [CitizenScience]   
4  28eb1a3f-1c15-4a95-931a-4af90ecb574d  [CitizenScience]   

                        installationKey                hostingOrganizationKey  \
0  997448a8-f762-11e1-a439-00145eb45e9a  28eb1a3f-1c15-4a95-931a-4af90ecb574d   
1  997448a8-f762-11e1-a439-00145eb45e9a  28eb1a3f-1c15-4a95-931a-4af90ecb574d   
2  99744